# PATHSNAP: Preprocessing & Candidate Generation Analysis
## Feature Extraction & Candidate Miss-Rate Evaluation

This notebook analyzes the results of candidate road generation, perpendicular metric distances, heading alignments, and candidate miss-rates.


### 1. Load Candidate Generation Summary


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

cand_summary = pd.read_csv('../outputs/reports/candidate_generation_summary.csv')
with open('../outputs/reports/preprocessing_summary.json') as f:
    prep_summary = json.load(f)

print("Preprocessing Execution Summary:")
print(json.dumps(prep_summary, indent=2))


### 2. Candidate Retrieval & Miss-Rate Evaluation
Evaluates whether the true ground-truth traversed arc is retrieved within the 50m search radius.


In [ ]:
print(f"Mean Candidate Hit Rate: {cand_summary['candidate_hit_rate'].mean() * 100:.2f}%")
print(f"Mean Candidate Miss Rate: {cand_summary['candidate_miss_rate'].mean() * 100:.2f}%")
print(f"Average Candidates per GPS Point: {cand_summary['mean_candidates_per_point'].mean():.2f}")
print(f"Zero-Candidate Points: {cand_summary['zero_candidate_points_count'].sum()}")

plt.figure(figsize=(10, 4))
sns.histplot(cand_summary['candidate_hit_rate'] * 100, bins=20, kde=True, color='#27ae60')
plt.title("Distribution of Candidate Hit Rate (%) Across Trips")
plt.xlabel("Candidate Hit Rate (%)")
plt.ylabel("Trip Count")
plt.show()


### 3. Inspection of Engineered Features for a Sample Trip


In [ ]:
sample_features = pd.read_csv('../data/features/train/00000001_candidates.csv')
print(f"Total candidate rows for Trip 00000001: {len(sample_features)}")
print("
Engineered Feature Columns:")
print(list(sample_features.columns))
sample_features.head(5)[['obs_id', 'candidate_arc_id', 'perp_distance', 'heading_diff', 'speed_kmh', 'emission_prob_gaussian', 'is_ground_truth']]


### 4. Partition Integrity & Leakage Prevention Audit
Verification that train (80 trips) and student_test (10 trips) are maintained strictly separate, and evaluator-test trips are held out.


In [ ]:
print("Partition Breakdown:")
print(cand_summary['partition'].value_counts())
train_trips = set(cand_summary[cand_summary['partition'] == 'train']['trip_id'])
test_trips = set(cand_summary[cand_summary['partition'] == 'student_test']['trip_id'])
assert len(train_trips.intersection(test_trips)) == 0, "DATA LEAKAGE DETECTED!"
print("Zero overlap between train and student_test partitions verified!")
